# Linear Regression & Optimization (CSC 422)

**In-class coding.** We start with nothing and end with a working linear
regression that trains itself. You write the model, the loss, the gradient and
the training loop — but never more than a line or two at a time.

**Duration:** 50 minutes

| | | you write |
|---|---|---|
| **0–6** | The problem | — |
| **6–16** | The model and the loss | `predict`, `loss` |
| **16–26** | Brute force, and why it dies | the grid score |
| **26–38** | The gradient, then one step | `gradients`, one update |
| **38–46** | The training loop | `fit` |
| **46–50** | One knob ruins everything | the learning rate |

---

### How the blanks work

Anywhere you see `______`, that is yours to fill. The answer is in the comment
on the same line, so you can walk through the whole session by reading down the
notebook. Fill it in, run it, look at the picture, move on.

`IC_2_linear_regression_instructor.ipynb` has every blank filled if you want to
run the whole thing beforehand.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(422)
plt.rcParams["figure.figsize"] = (7, 4.5)

---
## 0–6 min · The problem

Robby tracked cups of coffee against problems solved. **Where is the line?**

In [ ]:
x = np.random.uniform(0, 4, 40)
y = 2.5 * x - 1.0 + np.random.normal(0, 1.2, 40)     # the truth, plus noise

plt.scatter(x, y, s=45, alpha=.75, edgecolor="k", linewidth=.5)
plt.xlabel("cups of coffee"); plt.ylabel("problems solved")
plt.title("Where is the line?"); plt.grid(alpha=.2); plt.show()

**Ask the class:** *"Point at the slope. What is it, roughly?"*

They will say 2-ish. They just did regression in their heads. The computer does
not get to eyeball it — so we have to write down what they just did.

---
## 6–16 min · The model, and how we score it

A model is a rule with knobs. Ours has two: slope `a`, intercept `b`.

In [ ]:
def predict(a, b, x):
    return a * x + b

print(predict(2.0, 0.0, np.array([0, 1, 2])))   # expect [0. 2. 4.]

Now: how wrong is a given pair of knobs? Take every gap between the line and a
real point, square it so overshooting and undershooting both count, and average.

In [ ]:
def loss(a, b, x, y):
    return np.mean((y - predict(a, b, x)) ** 2)

print(f"a perfect fit on fake data: {loss(2.5, -1.0, np.array([1.0]), np.array([1.5])):.3f}")

**Two candidate lines. Which is better, and by how much?**

In [ ]:
print(f"Line A   y = 2.0x + 0.0    loss = {loss(2.0, 0.0, x, y):.3f}")
print(f"Line B   y = 3.0x - 2.0    loss = {loss(3.0, -2.0, x, y):.3f}")

In [ ]:
for a_, b_, c in [(2.0, 0.0, "crimson"), (3.0, -2.0, "teal")]:
    plt.plot(x, predict(a_, b_, x), color=c, lw=2,
             label=f"y={a_}x+{b_}   loss={loss(a_,b_,x,y):.2f}")
plt.scatter(x, y, s=40, alpha=.6, edgecolor="k", linewidth=.5, zorder=3)
plt.legend(); plt.grid(alpha=.2); plt.title("Two guesses, two numbers"); plt.show()

**The point.** "Which line looks better" is an opinion. **Loss makes it a
number** — and a number is something a computer can act on.

Everything from here is one question: *make that number small.*

---
## 16–26 min · Brute force: try every line

Two knobs. Lay a grid over both and score every combination.

In [ ]:
a_grid = np.linspace(-1, 5, 120)
b_grid = np.linspace(-4, 2, 120)
A_, B_ = np.meshgrid(a_grid, b_grid)

L = np.mean((y[:, None, None] - (A_ * x[:, None, None] + B_)) ** 2, axis=0)

i, j = np.unravel_index(L.argmin(), L.shape)
print(f"{L.size:,} lines tried")
print(f"best: a = {A_[i,j]:.2f},  b = {B_[i,j]:.2f},  loss = {L[i,j]:.3f}")

Now look at **the shape of what we just searched**. This picture is the course.

In [ ]:
from matplotlib.colors import LogNorm
lv = np.logspace(np.log10(L.min()), np.log10(L.max()), 40)

plt.contourf(A_, B_, L, levels=lv, norm=LogNorm(), cmap="viridis")
plt.colorbar(label="loss (log scale)")
plt.contour(A_, B_, L, levels=lv[::3], colors="white", linewidths=.4, alpha=.5)
plt.plot(A_[i,j], B_[i,j], "r*", ms=20, label="best found")
plt.xlabel("slope a"); plt.ylabel("intercept b")
plt.title("The loss surface — a long, narrow valley"); plt.legend(); plt.show()

In [ ]:
for knobs, name in [(2, "our line"), (100, "a small model"), (1_000_000, "a real network")]:
    print(f"{name:>16s}: {knobs:>9,} knobs  ->  10^{knobs:,} lines to check")

**The point.** Grid search dies instantly. Ten values per knob, two knobs, is
100 lines. A million knobs is $10^{1000000}$ — vastly more than there are atoms
in the universe.

We cannot look everywhere. **We have to start somewhere and walk downhill.**

---
## 26–38 min · Which way is down?

On the side of a bowl in the dark you do not need a map — only the slope under
your feet. Differentiating the loss gives one slope per knob:

$$\frac{\partial L}{\partial a} = \text{mean of } 2x(\hat y - y)
\qquad
\frac{\partial L}{\partial b} = \text{mean of } 2(\hat y - y)$$

In [ ]:
def gradients(a, b, x, y):
    err = predict(a, b, x) - y
    ga = 2 * np.mean(x * err)
    gb = 2 * np.mean(err)
    return ga, gb

print(gradients(0.0, 0.0, x, y))   # both negative: increase a and b

**Sanity check it.** Slice the bowl at `b = -1` and ask the gradient which way
is downhill from three places.

In [ ]:
aa = np.linspace(-1, 5, 200)
plt.plot(aa, [loss(a_, -1.0, x, y) for a_ in aa], color="0.6", lw=2)

for a_ in [0.0, 2.5, 4.5]:
    g, _ = gradients(a_, -1.0, x, y)
    plt.arrow(a_, loss(a_, -1, x, y), -g * 0.06, 0,
              head_width=1.6, head_length=.12, color="crimson", lw=2,
              length_includes_head=True)
    plt.plot(a_, loss(a_, -1, x, y), "ko", ms=7)
    plt.annotate(f"slope {g:+.1f}", (a_, loss(a_, -1, x, y)),
                 textcoords="offset points", xytext=(0, 12), ha="center", fontsize=9)

plt.xlabel("slope a"); plt.ylabel("loss"); plt.grid(alpha=.2)
plt.title("Arrow = $-$gradient. Long where steep, tiny at the bottom."); plt.show()

**Now take exactly one step.** Not a loop — one step, by hand, and watch the
loss drop.

In [ ]:
a, b = 0.0, 0.0
lr = 0.1

print(f"before:  a={a:.3f}  b={b:.3f}   loss={loss(a,b,x,y):.4f}")

ga, gb = gradients(a, b, x, y)
a = a - lr * ga
b = b - lr * gb

print(f"after :  a={a:.3f}  b={b:.3f}   loss={loss(a,b,x,y):.4f}")

**The point.** The loss went down. Not because we searched — because we asked
which way was down and stepped that way.

Do that 300 times and you have trained a model.

---
## 38–46 min · The training loop

In [ ]:
def fit(x, y, lr=0.1, steps=300):
    a, b = 0.0, 0.0                     # start anywhere
    path = [(a, b)]
    for _ in range(steps):
        ga, gb = gradients(a, b, x, y)  # which way is up?
        a -= lr * ga                    # step the other way
        b -= lr * gb
        path.append((a, b))
    return a, b, np.array(path)

a_hat, b_hat, path = fit(x, y)
print(f"grid search        a={A_[i,j]:.4f}  b={B_[i,j]:.4f}   loss={L[i,j]:.6f}   {L.size:,} lines")
print(f"gradient descent   a={a_hat:.4f}  b={b_hat:.4f}   loss={loss(a_hat,b_hat,x,y):.6f}      300 steps")

**The point.** That loop is the whole of machine learning. It beat a
14,400-point grid search in 300 steps — **and found a better line**, because a
grid can only ever land on its own gridpoints.

It also does not care whether there are 2 knobs or 2 billion. The transformer in
week 13 is trained by this exact loop.

In [ ]:
plt.contourf(A_, B_, L, levels=lv, norm=LogNorm(), cmap="viridis", alpha=.9)
plt.contour(A_, B_, L, levels=lv[::3], colors="white", linewidths=.4, alpha=.45)
plt.plot(path[:,0], path[:,1], "w.-", lw=1.6, ms=3, label="the walk")
plt.plot(0, 0, "wo", ms=11, mec="k", label="start")
plt.plot(a_hat, b_hat, "r*", ms=20, label="finish")
plt.xlabel("slope a"); plt.ylabel("intercept b")
plt.title("It rolled to the bottom"); plt.legend(loc="lower left"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), sharey=True)
for ax, step in zip(axes, [0, 3, 10, 300]):
    a_, b_ = path[step]
    ax.scatter(x, y, s=22, alpha=.5)
    ax.plot(x, predict(a_, b_, x), "crimson", lw=2.5)
    ax.set_title(f"step {step}   loss {loss(a_,b_,x,y):.2f}"); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

**Ask the class:** *"Between step 3 and step 10, what moved more — the slope or
the intercept?"*

Look back at the valley. It is steeper along `a` than along `b`, so the slope
gets fixed first and the intercept drifts in afterwards. Nobody told it to do
that.

---
## 46–50 min · One knob ruins everything

`lr` is how big a step we take — the one thing we never questioned. **Try the
three values below, then try your own.**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, lr in zip(axes, [0.005, 0.1, 0.55]):
    _, _, p = fit(x, y, lr=lr, steps=60)
    ax.plot([loss(a_, b_, x, y) for a_, b_ in p], lw=2, color="crimson")
    ax.set_title(f"lr = {lr}"); ax.set_xlabel("step"); ax.set_yscale("log"); ax.grid(alpha=.2)
axes[0].set_ylabel("loss (log scale)")
plt.tight_layout(); plt.show()

**The point.**

| | |
|---|---|
| `lr = 0.005` | correct, and uselessly slow |
| `lr = 0.1` | drops like a stone |
| `lr = 0.55` | **the loss goes up** — it steps over the bottom and climbs the far wall |

Same data, same code, same start. One number.

Not a toy failure: in **CA.01** a learning rate of 0.1 trains fine on scaled
features and sends the parameters to $10^{273}$ on raw ones. **PS1 problem 3**
is this by hand.

---
## What you built today

```python
def predict(a, b, x):      return a * x + b
def loss(a, b, x, y):      return np.mean((y - predict(a, b, x)) ** 2)
def gradients(a, b, x, y): err = predict(a,b,x) - y
                           return 2*np.mean(x*err), 2*np.mean(err)

for _ in range(steps):
    ga, gb = gradients(a, b, x, y)
    a -= lr * ga
    b -= lr * gb
```

Four ideas: **a model with knobs**, **a loss that scores it**, **a gradient that
points downhill**, **a step in the opposite direction**.

A neural network changes exactly one of them — the model gets bigger. Backprop
(Module 03) is just the chain rule for computing the gradient when there are
millions of knobs instead of two.

**Due Wednesday:** PS1, problem 3 is this lecture on paper.